# Phase 1 - Step 2: Data Validation

This notebook implements reusable pandas validation functions and validates all 5 raw datasets across schema, null percentages, ID uniqueness, range checks, and categorical integrity.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Validation Report Collector
validation_results = []

def record_check(dataset, check_name, status, details):
    validation_results.append({
        "dataset": dataset,
        "check_name": check_name,
        "status": status,
        "details": details
    })
    print(f"[{status}] {dataset} - {check_name}: {details}")

# Reusable Validation Functions
def validate_schema(df, expected_cols, dataset_name):
    missing_cols = [c for c in expected_cols if c not in df.columns]
    if missing_cols:
        record_check(dataset_name, "Schema Check", "FAIL", f"Missing expected columns: {missing_cols}")
    else:
        record_check(dataset_name, "Schema Check", "PASS", f"All {len(expected_cols)} expected columns present.")

def validate_duplicates(df, dataset_name):
    dups = df.duplicated().sum()
    if dups > 0:
        record_check(dataset_name, "Duplicate Rows", "WARNING", f"Found {dups} exact duplicate rows.")
    else:
        record_check(dataset_name, "Duplicate Rows", "PASS", "No duplicate rows found.")

def validate_nulls(df, max_null_pct, dataset_name):
    null_pct = df.isnull().mean() * 100
    high_nulls = null_pct[null_pct > max_null_pct]
    if not high_nulls.empty:
        record_check(dataset_name, "Null Percentage Check", "WARNING", f"Columns exceeding {max_null_pct}% nulls: {high_nulls.to_dict()}")
    else:
        record_check(dataset_name, "Null Percentage Check", "PASS", f"No columns exceed {max_null_pct}% null threshold.")

def validate_id_uniqueness(df, id_col, dataset_name):
    if id_col in df.columns:
        if df[id_col].nunique() == len(df):
            record_check(dataset_name, f"ID Uniqueness ({id_col})", "PASS", f"{id_col} is 100% unique.")
        else:
            record_check(dataset_name, f"ID Uniqueness ({id_col})", "FAIL", f"{id_col} has {len(df) - df[id_col].nunique()} non-unique entries.")

def validate_numeric_range(df, col, min_val, max_val, dataset_name):
    if col in df.columns:
        valid_mask = df[col].dropna().between(min_val, max_val)
        invalid_count = (~valid_mask).sum()
        if invalid_count == 0:
            record_check(dataset_name, f"Range Check ({col})", "PASS", f"All values within [{min_val}, {max_val}].")
        else:
            record_check(dataset_name, f"Range Check ({col})", "WARNING", f"{invalid_count} values outside [{min_val}, {max_val}].")

def validate_non_negative(df, num_cols, dataset_name):
    for col in num_cols:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            neg_count = (df[col] < 0).sum()
            if neg_count > 0:
                record_check(dataset_name, f"Non-negative Check ({col})", "FAIL", f"Found {neg_count} negative values in {col}.")
            else:
                record_check(dataset_name, f"Non-negative Check ({col})", "PASS", f"{col} has no negative values.")


In [2]:
# 1. Validate employee_attrition.csv
df_attr = pd.read_csv(raw_dir / "employee_attrition.csv")
validate_schema(df_attr, ['EmployeeID', 'Age', 'Department', 'JobRole', 'MonthlySalary', 'AttritionRisk'], "employee_attrition.csv")
validate_duplicates(df_attr, "employee_attrition.csv")
validate_nulls(df_attr, 20.0, "employee_attrition.csv")
validate_id_uniqueness(df_attr, "EmployeeID", "employee_attrition.csv")
validate_numeric_range(df_attr, "Age", 18, 100, "employee_attrition.csv")
validate_numeric_range(df_attr, "WorkLifeBalanceScore", 1.0, 5.5, "employee_attrition.csv")
validate_non_negative(df_attr, ['MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'YearsAtCompany'], "employee_attrition.csv")
if 'AttritionRisk' in df_attr.columns:
    valid_cats = {'Yes', 'No'}
    actual_cats = set(df_attr['AttritionRisk'].dropna().unique())
    if actual_cats.issubset(valid_cats):
        record_check("employee_attrition.csv", "Attrition Category Check", "PASS", f"Categories valid: {actual_cats}")
    else:
        record_check("employee_attrition.csv", "Attrition Category Check", "FAIL", f"Unexpected categories: {actual_cats - valid_cats}")

# 2. Validate hr_performance_engagement.csv
df_eng = pd.read_csv(raw_dir / "hr_performance_engagement.csv")
validate_schema(df_eng, ['Employee ID', 'Department', 'Job Role', 'Performance Score', 'KPI Score', 'Attendance (%)'], "hr_performance_engagement.csv")
validate_duplicates(df_eng, "hr_performance_engagement.csv")
validate_nulls(df_eng, 5.0, "hr_performance_engagement.csv")
validate_id_uniqueness(df_eng, "Employee ID", "hr_performance_engagement.csv")
validate_numeric_range(df_eng, "Attendance (%)", 0, 100, "hr_performance_engagement.csv")
validate_numeric_range(df_eng, "Task Completion (%)", 0, 100, "hr_performance_engagement.csv")
validate_non_negative(df_eng, ['Performance Score', 'Work Hours Logged', 'Training Hours'], "hr_performance_engagement.csv")

# 3. Validate occupation_data.csv
df_occ = pd.read_csv(raw_dir / "occupation_data.csv")
validate_schema(df_occ, ['O*NET-SOC Code', 'Title', 'Description'], "occupation_data.csv")
validate_duplicates(df_occ, "occupation_data.csv")
validate_nulls(df_occ, 0.0, "occupation_data.csv")
validate_id_uniqueness(df_occ, "O*NET-SOC Code", "occupation_data.csv")

# 4. Validate essential_skills.csv
df_ess = pd.read_csv(raw_dir / "essential_skills.csv")
validate_schema(df_ess, ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value'], "essential_skills.csv")
validate_duplicates(df_ess, "essential_skills.csv")
validate_nulls(df_ess, 50.0, "essential_skills.csv")
validate_numeric_range(df_ess, "Data Value", 0, 100, "essential_skills.csv")

# 5. Validate software_skills.csv
df_soft = pd.read_csv(raw_dir / "software_skills.csv")
validate_schema(df_soft, ['O*NET-SOC Code', 'Title', 'Workplace Example'], "software_skills.csv")
validate_duplicates(df_soft, "software_skills.csv")
validate_nulls(df_soft, 0.0, "software_skills.csv")


[PASS] employee_attrition.csv - Schema Check: All 6 expected columns present.
[PASS] employee_attrition.csv - Duplicate Rows: No duplicate rows found.
[WARNING] employee_attrition.csv - Null Percentage Check: Columns exceeding 20.0% nulls: {'CustomerSatisfaction': 63.800000000000004}
[PASS] employee_attrition.csv - ID Uniqueness (EmployeeID): EmployeeID is 100% unique.
[PASS] employee_attrition.csv - Range Check (Age): All values within [18, 100].
[WARNING] employee_attrition.csv - Range Check (WorkLifeBalanceScore): 250 values outside [1.0, 5.5].
[PASS] employee_attrition.csv - Non-negative Check (MonthlySalary): MonthlySalary has no negative values.
[PASS] employee_attrition.csv - Non-negative Check (OvertimeHoursPerMonth): OvertimeHoursPerMonth has no negative values.
[PASS] employee_attrition.csv - Non-negative Check (LeavesTaken): LeavesTaken has no negative values.
[PASS] employee_attrition.csv - Non-negative Check (YearsAtCompany): YearsAtCompany has no negative values.
[PASS] e

[PASS] software_skills.csv - Duplicate Rows: No duplicate rows found.
[PASS] software_skills.csv - Null Percentage Check: No columns exceed 0.0% null threshold.


In [3]:
report_df = pd.DataFrame(validation_results)
report_csv_path = processed_dir / "data_validation_report.csv"
report_df.to_csv(report_csv_path, index=False)
print(f"\nSaved validation report to {report_csv_path}")
print(f"Status Summary:\n{report_df['status'].value_counts()}")
print(report_df.to_string())



Saved validation report to data\processed\data_validation_report.csv
Status Summary:
status
PASS       29
WARNING     2
Name: count, dtype: int64
                          dataset                                  check_name   status                                                                      details
0          employee_attrition.csv                                Schema Check     PASS                                              All 6 expected columns present.
1          employee_attrition.csv                              Duplicate Rows     PASS                                                     No duplicate rows found.
2          employee_attrition.csv                       Null Percentage Check  WARNING  Columns exceeding 20.0% nulls: {'CustomerSatisfaction': 63.800000000000004}
3          employee_attrition.csv                  ID Uniqueness (EmployeeID)     PASS                                                   EmployeeID is 100% unique.
4          employee_attrition.csv